## Charger les données


In [4]:
import pandas as pd
from pathlib import Path


bronze_path = Path.cwd().parent / "data" / "bronze"
# Charger les données Bronze
df_items = pd.read_csv(bronze_path / "olist_order_items_dataset.csv")


# Afficher les 5 premières lignes
df_items.head(5)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


## Vérification:  la structure

In [5]:
# Nombre de lignes et de colonnes
print("Dimensions :", df_items.shape)

# Types des colonnes
print("\nTypes des données :")
print(df_items.dtypes)

# Informations générales
df_items.info()

Dimensions : (112650, 7)

Types des données :
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


## Vérification les dates

In [6]:
# Convertir les dates sans modifier la colonne originale
dates_converties = pd.to_datetime(
    df_items["shipping_limit_date"],
    errors="coerce"
)

# Vérifier les erreurs de conversion
print(
    "Dates invalides :",
    dates_converties.isna().sum()
)

# Afficher la date minimale et maximale
print("Date minimale :", dates_converties.min())
print("Date maximale :", dates_converties.max())

Dates invalides : 0
Date minimale : 2016-09-19 00:15:34
Date maximale : 2020-04-09 22:35:08


## Vérification des doublons

In [7]:
# Vérifier les lignes entièrement identiques
print("Lignes dupliquées :", df_items.duplicated().sum())

# Vérifier les doublons sur la clé composée
print(
    "Clés (order_id, order_item_id) dupliquées :",
    df_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

# Vérifier combien de commandes contiennent plusieurs articles
print(
    "Commandes avec plusieurs articles :",
    (df_items.groupby("order_id").size() > 1).sum()
)

Lignes dupliquées : 0
Clés (order_id, order_item_id) dupliquées : 0
Commandes avec plusieurs articles : 9803


##  Vérification des valeurs numériques

In [8]:
# Statistiques des prix et frais de livraison
df_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [9]:
# Vérifier les montants négatifs
print("Prix négatifs :", (df_items["price"] < 0).sum())

print("Frais de livraison négatifs :",
      (df_items["freight_value"] < 0).sum())

# Vérifier les montants à zéro
print("Prix à zéro :", (df_items["price"] == 0).sum())

print("Frais de livraison à zéro :",
      (df_items["freight_value"] == 0).sum())

Prix négatifs : 0
Frais de livraison négatifs : 0
Prix à zéro : 0
Frais de livraison à zéro : 383


In [10]:
# Vérifier la numérotation des articles dans chaque commande
numerotation_invalide = (
    df_items.groupby("order_id")["order_item_id"]
    .apply(lambda x: sorted(x.tolist()) != list(range(1, len(x) + 1)))
)

print(
    "Commandes avec numérotation incorrecte :",
    numerotation_invalide.sum()
)

Commandes avec numérotation incorrecte : 0
